# Working With Views
Demonstrates creating, querying, and dropping **persistent** and **temporary** views in Databricks using Spark SQL.

| View Type | Scope | Stored in Metastore |
|-----------|-------|---------------------|
| Persistent | Survives session restarts | Yes |
| Temporary | Current session only | No |
| Global Temporary | All sessions in the app | No (`global_temp` db) |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## Persistent View
A persistent view is stored in the metastore and survives session restarts.
It behaves like a virtual table defined by a query — no data is copied.
Similar to the SQL Server view although there is present MATERIALIZED options, and we could do indexing on top of that. 

In [ ]:
spark.sql("""
    CREATE OR REPLACE VIEW default.vw_top10_countries_by_population AS
    SELECT *
    FROM default.country_population
    ORDER BY POPULATION DESC
    LIMIT 10
""")

print("Persistent view created: vw_top10_countries_by_population")

In [ ]:
spark.sql("""
    SELECT *
    FROM default.vw_top10_countries_by_population
""").show(truncate=False)

## Temporary View
A temporary view is **session-scoped** — it exists only for the duration of the current Spark session
and is not stored in the metastore. No cleanup is required; it is dropped automatically when the session ends.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW tmp_top10_countries AS
    SELECT *
    FROM default.country_population
    ORDER BY POPULATION DESC
    LIMIT 10
""")

print("Temporary view created: tmp_top10_countries")

spark.sql("SELECT * FROM tmp_top10_countries").show(truncate=False)

## Showing Views
Use `SHOW VIEWS` to list all views in the current schema. Temporary views appear with `isTemporary = true`.

In [ ]:
spark.sql("""
    SHOW VIEWS IN default
""").show(truncate=False)

## Drop the Persistent View
Persistent views must be explicitly dropped with `DROP VIEW`. `IF EXISTS` prevents an error if the view has already been removed.

In [ ]:
spark.sql("""
    DROP VIEW IF EXISTS default.vw_top10_countries_by_population
""")

print("Persistent view dropped.")

In [ ]:
# After dropping the persistent view, only the temporary view should remain
spark.sql("""
    SHOW VIEWS IN default
""").show(truncate=False)

## Materialized View (Delta Live Tables)
A **materialized view** physically stores query results — unlike a regular view which re-executes the query on every read.
In Databricks, materialized views are a feature of **Delta Live Tables (DLT)** pipelines.

| Feature | Regular View | Materialized View (DLT) |
|---------|-------------|-------------------------|
| Storage | None — query runs on each access | Yes — results stored as a Delta table |
| Refresh | Always up-to-date | Updated when pipeline runs |
| Performance | Depends on source query cost | Fast reads from pre-computed results |
| Context | Any notebook / SQL | DLT pipeline only |

In the `SHOW VIEWS` output the column `isMaterialized` distinguishes them from regular views.

> **Note:** The code below uses the `dlt` library and must run inside a **DLT pipeline notebook**.
> Running it in a standard interactive notebook will raise `ImportError` — this is expected.

In [ ]:
# NOTE: This cell must run inside a Delta Live Tables (DLT) pipeline.
# In a regular interactive notebook it will raise ImportError — that is expected.

import dlt

@dlt.table(
    name="mv_top10_countries_by_population",
    comment="Materialized view — top 10 most populated countries, refreshed by the DLT pipeline"
)
def mv_top10_countries_by_population():
    return spark.sql("""
        SELECT *
        FROM default.country_population
        ORDER BY POPULATION DESC
        LIMIT 10
    """)

# The same materialized view expressed as DLT SQL (in a DLT SQL notebook):
#
# CREATE OR REFRESH MATERIALIZED VIEW mv_top10_countries_by_population AS
# SELECT *
# FROM country_population
# ORDER BY POPULATION DESC
# LIMIT 10

## SELECT INTO Equivalent — Writing a DataFrame to a Delta Table
In SQL Server `SELECT INTO` creates a new physical table from a query result in one step:

```sql
-- SQL Server
SELECT * INTO dbo.top10_countries_snapshot
FROM dbo.country_population
ORDER BY POPULATION DESC
```

In Databricks the equivalent is **CTAS** (Create Table As Select).
The result is written as a **Delta table** — a real, persisted table, not a view.

```sql
-- Databricks SQL
CREATE OR REPLACE TABLE catalog.schema.new_table AS
SELECT * FROM source_table ...
```

The DataFrame write API achieves the same result in Python style:

```python
df.write.format("delta").mode("overwrite").saveAsTable("catalog.schema.new_table")
```

Both approaches:
- physically store the data on disk (Delta format under the hood)
- support all subsequent DML operations (`INSERT`, `UPDATE`, `DELETE`, `MERGE`)
- appear in the metastore as a regular table, not as a view

In [ ]:
# ── Method 1: SQL CTAS — closest equivalent to SQL Server SELECT INTO ─────────
spark.sql("""
    CREATE OR REPLACE TABLE default.top10_countries_snapshot AS
    SELECT *
    FROM default.country_population
    ORDER BY POPULATION DESC
    LIMIT 10
""")

print("Table created via CTAS: top10_countries_snapshot")

# ── Method 2: DataFrame write API — same result, Python style ─────────────────
# df = spark.sql("""
#     SELECT *
#     FROM default.country_population
#     ORDER BY POPULATION DESC
#     LIMIT 10
# """)
# df.write.format("delta").mode("overwrite").saveAsTable(
#     "default.top10_countries_snapshot"
# )

In [ ]:
# Verify the snapshot table — this is real, persisted data (not a view)
spark.sql("""
    SELECT * FROM default.top10_countries_snapshot
""").show(truncate=False)

# DESCRIBE to confirm it is a TABLE (not a view) and stored in Delta format
spark.sql("""
    DESCRIBE EXTENDED default.top10_countries_snapshot
""").filter("col_name IN ('Type', 'Provider', 'Location')").show(truncate=False)

In [ ]:
spark.sql("""
    DROP TABLE IF EXISTS default.top10_countries_snapshot
""")

print("Snapshot table dropped.")